In [11]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import json

In [12]:
df = pd.read_csv('C:/Users/user/JobGenie/Student/Dataset/raw/student_salary_dataset.csv')
print(f"✅ Loaded: {df.shape[0]} rows × {df.shape[1]} columns")

✅ Loaded: 5000 rows × 14 columns


In [13]:
df.drop(columns=['skills'], inplace=True)
tier_map = {'Tier 1': 3, 'Tier 2': 2, 'Tier 3': 1}
df['college_tier_encoded'] = df['college_tier'].map(tier_map)
le = LabelEncoder()
df['job_role_encoded'] = le.fit_transform(df['job_role'])
df['aptitude_score'] = df['aptitude_score'].round(1)
 
print("✅ Cleaning reproduced")
print()

✅ Cleaning reproduced



In [14]:

# ─── DECISION 1: KEEP ALL WEAK FEATURES ──────────────────────────────────────
# Analysis showed 5 features have low individual correlation with salary:
#   internships (0.03), github_projects (0.05), hackathons (0.04),
#   certifications (0.01), basic_skills (0.05)
#
# We are KEEPING all of them for two reasons:
#   1. Random Forest captures non-linear interactions — a feature weak in
#      isolation can still be useful in combination (e.g. internships +
#      advanced_skills together might signal a strong candidate).
#   2. For the improvement suggestions system in Flask, we need these values
#      to give advice like "do more hackathons" — dropping them breaks that.
#
# What we DO NOT do: drop features just because correlation is low.
# Correlation only measures linear relationships. Tree models see more.
 
print("─" * 55)
print("DECISION 1: Keeping all weak-correlation features")
print("  Reason: RF captures non-linear interactions.")
print("  Kept: internships, github_projects, hackathons,")
print("        certifications, basic_skills")

───────────────────────────────────────────────────────
DECISION 1: Keeping all weak-correlation features
  Reason: RF captures non-linear interactions.
  Kept: internships, github_projects, hackathons,
        certifications, basic_skills


In [15]:

# ─── DECISION 2: HANDLE cgpa ↔ aptitude_score MULTICOLLINEARITY ──────────────
# These two features have correlation = 0.816 — very high.
# High multicollinearity can confuse linear models (they can't tell which
# feature to credit). For Random Forest it's less critical but still worth
# addressing.
#
# Options considered:
#   A) Drop one → Bad idea. Both are strongly correlated with salary (0.73, 0.74)
#      and dropping either loses real signal.
#   B) PCA → Overkill for 2 features, hard to explain in a college project.
#   C) Create a combined score → Best option here.
#
# We create: academic_score = weighted combination of normalized CGPA + aptitude
# This captures both signals in one feature, removing the redundancy.
# Weight: 60% CGPA, 40% aptitude (CGPA matters more in Indian placements).
 
cgpa_norm = (df['cgpa'] - df['cgpa'].min()) / (df['cgpa'].max() - df['cgpa'].min())
apt_norm  = (df['aptitude_score'] - df['aptitude_score'].min()) / (df['aptitude_score'].max() - df['aptitude_score'].min())
df['academic_score'] = (0.6 * cgpa_norm + 0.4 * apt_norm).round(4)
 
print()
print("─" * 55)
print("DECISION 2: Created academic_score (cgpa + aptitude combined)")
print(f"  Formula : 0.6 × norm_cgpa + 0.4 × norm_aptitude")
print(f"  Range   : {df['academic_score'].min():.3f} – {df['academic_score'].max():.3f}")
print(f"  Corr with salary: {df['academic_score'].corr(df['salary_lpa']):.4f}")
print(f"  (vs cgpa alone: {df['cgpa'].corr(df['salary_lpa']):.4f})")
 


───────────────────────────────────────────────────────
DECISION 2: Created academic_score (cgpa + aptitude combined)
  Formula : 0.6 × norm_cgpa + 0.4 × norm_aptitude
  Range   : 0.000 – 1.000
  Corr with salary: 0.7587
  (vs cgpa alone: 0.7315)


In [16]:
# ─── DECISION 3: CREATE total_skills FEATURE ─────────────────────────────────
# Currently skills are split into 3 separate counts:
#   advanced_skills, intermediate_skills, basic_skills
# We create a weighted total_skills that gives more credit to advanced skills.
# This is a more meaningful single number than just summing all equally.
#
# Weights based on salary impact from EDA:
#   advanced     → each adds ~3-5 LPA   → weight 3
#   intermediate → each adds ~1-2 LPA   → weight 2
#   basic        → minimal salary impact → weight 1
 
df['total_skills'] = (
    df['advanced_skills'] * 3 +
    df['intermediate_skills'] * 2 +
    df['basic_skills'] * 1
)
 
print()
print("─" * 55)
print("DECISION 3: Created total_skills (weighted skill score)")
print(f"  Formula : (adv×3) + (inter×2) + (basic×1)")
print(f"  Range   : {df['total_skills'].min()} – {df['total_skills'].max()}")
print(f"  Corr with salary: {df['total_skills'].corr(df['salary_lpa']):.4f}")
print(f"  (vs advanced_skills alone: {df['advanced_skills'].corr(df['salary_lpa']):.4f})")


───────────────────────────────────────────────────────
DECISION 3: Created total_skills (weighted skill score)
  Formula : (adv×3) + (inter×2) + (basic×1)
  Range   : 1 – 15
  Corr with salary: 0.5000
  (vs advanced_skills alone: 0.5041)


In [17]:
# ─── DECISION 4: CREATE experience_score FEATURE ─────────────────────────────
# internships, github_projects, hackathons, certifications all have weak
# individual correlations but together represent "overall experience effort".
# We create a single experience_score combining all four.
#
# Weights:
#   internships     → highest weight (real work experience)
#   github_projects → second (shows actual coding output)
#   hackathons      → third (shows initiative)
#   certifications  → least (easy to get, less impressive)
 
df['experience_score'] = (
    df['internships']     * 3 +
    df['github_projects'] * 2 +
    df['hackathons']      * 1.5 +
    df['certifications']  * 1
).round(2)
 
print()
print("─" * 55)
print("DECISION 4: Created experience_score (combined experience)")
print(f"  Formula : (intern×3) + (github×2) + (hack×1.5) + (cert×1)")
print(f"  Range   : {df['experience_score'].min()} – {df['experience_score'].max()}")
print(f"  Corr with salary: {df['experience_score'].corr(df['salary_lpa']):.4f}")
 



───────────────────────────────────────────────────────
DECISION 4: Created experience_score (combined experience)
  Formula : (intern×3) + (github×2) + (hack×1.5) + (cert×1)
  Range   : 0.0 – 29.0
  Corr with salary: 0.0484


In [18]:
# ─── DECISION 5: NO SCALING NEEDED FOR TREE MODELS ──────────────────────────
# Random Forest does NOT require feature scaling.
# Trees split on thresholds — the scale of a feature doesn't matter.
# Scaling is only needed for: Linear Regression, SVM, KNN, Neural Networks.
#
# We will NOT apply StandardScaler or MinMaxScaler to the training features.
# This simplifies the Flask API (no need to scale incoming data before predict).
 
print()
print("─" * 55)
print("DECISION 5: No scaling applied")
print("  Reason: Random Forest is scale-invariant.")
print("  Benefit: Flask API doesn't need to scale input data.")


───────────────────────────────────────────────────────
DECISION 5: No scaling applied
  Reason: Random Forest is scale-invariant.
  Benefit: Flask API doesn't need to scale input data.


In [19]:
# ─── DECISION 6: DEFINE TWO FEATURE SETS ─────────────────────────────────────
# We create two sets — one with engineered features, one without.
# During model training (Step 5) we test both and pick the better one.
 
FEATURES_ORIGINAL = [
    'college_tier_encoded',
    'cgpa',
    'internships',
    'github_projects',
    'backlogs',
    'hackathons',
    'certifications',
    'advanced_skills',
    'intermediate_skills',
    'basic_skills',
    'aptitude_score'
]
 
FEATURES_ENGINEERED = [
    'college_tier_encoded',
    'academic_score',      # replaces cgpa + aptitude_score
    'total_skills',        # replaces advanced + intermediate + basic
    'experience_score',    # replaces internships + github + hackathons + certs
    'backlogs',            # kept as-is (negative signal, important alone)
]
 
print()
print("─" * 55)
print("DECISION 6: Two feature sets defined for comparison")
print(f"  Original set   : {len(FEATURES_ORIGINAL)} features")
print(f"  Engineered set : {len(FEATURES_ENGINEERED)} features")



───────────────────────────────────────────────────────
DECISION 6: Two feature sets defined for comparison
  Original set   : 11 features
  Engineered set : 5 features


In [20]:
# ─── QUICK COMPARISON ────────────────────────────────────────────────────────
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
 
X_orig = df[FEATURES_ORIGINAL]
X_eng  = df[FEATURES_ENGINEERED]
y      = df['salary_lpa']
 
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
 
scores_orig = cross_val_score(rf, X_orig, y, cv=5, scoring='r2')
scores_eng  = cross_val_score(rf, X_eng,  y, cv=5, scoring='r2')
 
print()
print("─" * 55)
print("QUICK CV COMPARISON (R² score, 5-fold):")
print(f"  Original features  : {scores_orig.mean():.4f} ± {scores_orig.std():.4f}")
print(f"  Engineered features: {scores_eng.mean():.4f} ± {scores_eng.std():.4f}")
 
winner = "ORIGINAL" if scores_orig.mean() >= scores_eng.mean() else "ENGINEERED"
print(f"  → Winner: {winner} features — will use in model training")
 
# Pick the winning feature set
FINAL_FEATURES = FEATURES_ORIGINAL if winner == "ORIGINAL" else FEATURES_ENGINEERED
 
# ─── SAVE FINAL DATASET ──────────────────────────────────────────────────────
df.to_csv('C:/Users/user/JobGenie/Student/Dataset/student_featured.csv', index=False)
 
meta = {
    'features_original'  : FEATURES_ORIGINAL,
    'features_engineered': FEATURES_ENGINEERED,
    'final_features'     : FINAL_FEATURES,
    'target_salary'      : 'salary_lpa',
    'target_role'        : 'job_role_encoded',
    'role_classes'       : list(le.classes_),
    'tier_mapping'       : tier_map,
    'role_mapping'       : {k: int(v) for k, v in
                            zip(le.classes_, le.transform(le.classes_))},
    'winner'             : winner
}
with open('C:/Users/user/JobGenie/Student/Dataset/feature_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
 
print()
print("=" * 55)
print("  FEATURE ENGINEERING COMPLETE")
print("=" * 55)
print(f"  New features created:")
print(f"    ✓ academic_score   (cgpa + aptitude combined)")
print(f"    ✓ total_skills     (weighted skill score)")
print(f"    ✓ experience_score (internship + github + hack + cert)")
print(f"  Final feature set   : {winner}")
print(f"  Features going into model: {FINAL_FEATURES}")
print(f"  Saved: student_featured.csv")
print(f"  Saved: feature_meta.json")
print("=" * 55)
 


───────────────────────────────────────────────────────
QUICK CV COMPARISON (R² score, 5-fold):
  Original features  : 0.9252 ± 0.0038
  Engineered features: 0.7269 ± 0.0154
  → Winner: ORIGINAL features — will use in model training

  FEATURE ENGINEERING COMPLETE
  New features created:
    ✓ academic_score   (cgpa + aptitude combined)
    ✓ total_skills     (weighted skill score)
    ✓ experience_score (internship + github + hack + cert)
  Final feature set   : ORIGINAL
  Features going into model: ['college_tier_encoded', 'cgpa', 'internships', 'github_projects', 'backlogs', 'hackathons', 'certifications', 'advanced_skills', 'intermediate_skills', 'basic_skills', 'aptitude_score']
  Saved: student_featured.csv
  Saved: feature_meta.json
